# Reference / test run notebook for any model

This notebook is used to test a **newly added model** trough these 8 steps:
1. Load the config file and inspect which model and output keys it points to
2. Validate that the model API is complete
3. Run one reference / test simulation
4. Check whether output files were created
5. Read PK/PD trajectories from the HDF5 result file using the YAML-defined keys
6. Plot the last dosing interval
7. Recompute summary metrics and compare them with the saved JSON summary
8. Optionally test numerical robustness to `dt_days`, solver method, and tolerances

## Minimal requirements for a new model

### `models/<NewModel>.py`
The model file must expose:
- `DEFAULTS`
- `validate_params(params)`
- `initial_conditions(params)`
- `apply_dose(y, dose_mgkg, params)`
- `rhs(t, y, params)`
- `derived(t, y, params)`

### `configs/<NewModel>_sweep.yaml`
The config file must contain at least:
- `model`
- `params`
- `simulation`
- `outputs`
- `output`


## Imports and notebook setup

### Inputs
- **`CONFIG_PATH`**: path to the YAML config of the model you want to test
- **`DOSE_MGKG`**: test dose in mg/kg
- **`INTERVAL_WEEKS`**: dosing interval in weeks
- **`PARAM_OVERRIDES`**: optional dictionary with parameter overrides for a quick test run
- **`OVERWRITE`**:
  - `True` → rerun the simulation and overwrite existing files
  - `False` → reuse an existing result if available

### Expected output
This section should print the detected repository root.

### If something goes wrong
- If imports fail, the notebook is probably not started from the expected repo location.
- If `scripts.runner` cannot be imported, check that the repo contains `configs/`, `scripts/`, and `models/` and that you are running inside the project.


In [ ]:
# -------------------------
# INPUT — edit only here
# -------------------------
from pathlib import Path

CONFIG_PATH = Path('configs/<ModelName>_sweep.yaml')
DOSE_MGKG = 3.0
INTERVAL_WEEKS = 2
OVERWRITE = True

# Optional parameter overrides for quick testing, e.g. {'Cl_ml_day': 150}
PARAM_OVERRIDES = None

# -------------------------
# IMPORTS & REPO ROOT
# -------------------------

import sys
import json
import importlib
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import sys

# -------------------------
# Resolve repository root
# -------------------------

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "scripts" / "runner.py").exists()
            and (candidate / "configs").exists()
            and (candidate / "models").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate repository root. Please run this notebook from inside the repo "
        "or check that scripts/runner.py, configs/, and models/ exist."
    )

repo_root = find_repo_root()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
from scripts.runner import run_reference, run_one, run_sweep, load_config


## 1. Load config and inspect the model target

This cell loads the YAML config and prints the key fields used by the workflow:
- model name
- model module import path
- PK output key
- PD output key

It also displays the full parsed config dictionary.

### What you should check
- `model.module` points to the correct Python file in `models/`
- `pk_key` matches a key that will be returned by `derived(...)`
- `pd_key` is either correct or intentionally `None`
- simulation and output settings look reasonable


### If something goes wrong
- **File not found** → `CONFIG_PATH` is wrong
- **YAML parsing error** → formatting issue in the config file
- **Wrong `model.module`** → the later import step will fail even if the YAML loads here


In [ ]:
config_path = CONFIG_PATH if CONFIG_PATH.is_absolute() else (repo_root / CONFIG_PATH)
cfg = load_config(config_path)

model_name = cfg['model']['name']
model_module_name = cfg['model']['module']
pk_key = cfg.get('outputs', {}).get('pk_key', 'Central_ugml')
pd_key = cfg.get('outputs', {}).get('pd_key', None)

print('Config path :', config_path)
print('Model name  :', model_name)
print('Module      :', model_module_name)
print('PK key      :', pk_key)
print('PD key      :', pd_key)

cfg


## 2. Validate the model API

This cell imports the model module and checks whether it exposes the required shared interface used by `runner.py`.
It then validates the effective parameter set built from:
- model `DEFAULTS`
- YAML `params`
- optional `PARAM_OVERRIDES`


### If something goes wrong
Common causes:
- a required function is missing from the model file
- a required parameter is missing from `DEFAULTS` and YAML `params`
- the YAML uses parameter names that the model does not expect
- the model import path in the YAML is wrong

This is one of the most useful cells in the notebook, because it catches integration errors early, before you spend time debugging solver output.


In [ ]:
model_module = importlib.import_module(model_module_name)

required_attributes = [
    'DEFAULTS',
    'validate_params',
    'initial_conditions',
    'apply_dose',
    'rhs',
    'derived',
]

missing = [name for name in required_attributes if not hasattr(model_module, name)]
if missing:
    raise AttributeError(
        f"Model module '{model_module_name}' is missing required attributes: {missing}"
    )

# Validate parameters using DEFAULTS + cfg.params
params = dict(getattr(model_module, 'DEFAULTS', {}))
params.update(cfg.get('params', {}))
if PARAM_OVERRIDES:
    params.update(PARAM_OVERRIDES)

model_module.validate_params(params)

print('Model API check passed.')
print('Required attributes found:', required_attributes)
print('Parameter validation passed.')


## 3. Run one reference / test simulation

This cell calls `run_reference(...)` from the shared runner.
The runner:
- imports the model defined in the YAML
- builds the effective parameter set
- runs the simulation
- writes result files
- returns a summary dictionary

### Expected output
A summary dictionary with values such as:
- `C_trough_ugml`
- `C_avg_ugml`
- `C_max_ugml`
- optionally `<pd_key>_trough`
- `_run_dir`

### If something goes wrong
Typical causes:
- ODE system error inside `rhs(...)`
- invalid initial conditions
- invalid parameter values
- `derived(...)` missing the configured output keys
- numerical instability for the chosen settings


In [ ]:
summary = run_reference(
    config_path=config_path,
    dose_mgkg=DOSE_MGKG,
    interval_weeks=INTERVAL_WEEKS,
    param_overrides=PARAM_OVERRIDES,
    overwrite=OVERWRITE,
)

summary


## 4. Inspect generated output files

This cell checks whether the expected files were written to the run directory:
- `run_summary.json`
- `run.h5`
- `run_config.json`

### What you should expect
All three existence checks should usually return `True`.
The JSON summary is then loaded and displayed.

### Why this matters
If the simulation finished but one of these files is missing, the problem is no longer the model equations themselves. It is then more likely an output-writing or configuration issue.

### If something goes wrong
- missing `run.h5` → simulation or file writing did not complete correctly
- missing `run_summary.json` → summary export failed or the run stopped early
- missing `run_config.json` → output-writing settings in the config may have been changed


In [ ]:
run_dir = Path(summary['_run_dir'])
summary_path = run_dir / 'run_summary.json'
h5_path = run_dir / 'run.h5'
run_config_json_path = run_dir / 'run_config.json'

print('Run dir            :', run_dir)
print('Summary exists     :', summary_path.exists())
print('H5 exists          :', h5_path.exists())
print('Config JSON exists :', run_config_json_path.exists())

with open(summary_path, 'r', encoding='utf-8') as f:
    summary_json = json.load(f)

summary_json


## 5. Read PK/PD trajectories from HDF5 using YAML-defined output keys

This cell reads the **last dosing interval** from `run.h5`.
It loads:
- the time grid `t_days`
- the PK trajectory from `outputs.pk_key`
- the PD trajectory from `outputs.pd_key` if a PD key is defined

### What you should expect
You should see:
- the simulated time range for the last interval
- the number of time points
- the list of available derived keys stored in the HDF5 file


### If something goes wrong
- **PK key not found** → `outputs.pk_key` in YAML does not match the name returned by `derived(...)`
- **PD key not found** → same issue for `outputs.pd_key`
- **missing `derived` group** → the run file may be incomplete or corrupted


In [ ]:
with h5py.File(h5_path, 'r') as f:
    t_days = f['t_days'][:]
    derived_group = f['derived']

    if pk_key not in derived_group:
        raise KeyError(f"PK key '{pk_key}' not found in HDF5. Available: {list(derived_group.keys())}")
    pk_values = derived_group[pk_key][:]

    if pd_key is not None:
        if pd_key not in derived_group:
            raise KeyError(f"PD key '{pd_key}' not found in HDF5. Available: {list(derived_group.keys())}")
        pd_values = derived_group[pd_key][:]
    else:
        pd_values = None

print('Time range (days):', float(t_days[0]), '->', float(t_days[-1]))
print('Number of time points:', len(t_days))
print('Available derived keys:', list(h5py.File(h5_path, 'r')['derived'].keys()))


## 6. Plot the last dosing interval

This cell plots the trajectories that were loaded from the HDF5 file.
- If a PD key is defined, you get two panels: PK and PD
- If no PD key is defined, only the PK panel is shown

### What you should expect
A quick visual sanity check of the simulated outputs.
This is often the fastest way to spot problems such as:
- exploding trajectories
- negative values where they should not occur
- flat lines caused by missing dosing or a broken effect model
- obvious scale issues from wrong units

### If something looks wrong
Check:
- parameter units
- dose handling in `apply_dose(...)`
- compartment or state definitions in `initial_conditions(...)`
- equations in `rhs(...)`
- formulas and naming in `derived(...)`


In [ ]:
if pd_values is None:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(t_days, pk_values)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel(pk_key)
    ax.set_title(f'{model_name}: {pk_key} – last dosing interval')
    ax.grid(True)
    plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(t_days, pk_values)
    axes[0].set_xlabel('Time (days)')
    axes[0].set_ylabel(pk_key)
    axes[0].set_title(f'{model_name}: {pk_key} – last dosing interval')
    axes[0].grid(True)

    axes[1].plot(t_days, pd_values)
    axes[1].set_xlabel('Time (days)')
    axes[1].set_ylabel(pd_key)
    axes[1].set_title(f'{model_name}: {pd_key} – last dosing interval')
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


## 7. Recompute summary metrics from arrays and compare with the saved summary

This cell recalculates summary values directly from the loaded trajectories and compares them with the saved values from `run_summary.json`.

The main checks are:
- `C_trough_ugml`
- `C_avg_ugml`
- `C_max_ugml`
- optionally `<pd_key>_trough`

### What you should expect
The `saved` and `recomputed` columns should be identical or extremely close.
Small floating-point differences can be normal.

### Why this matters
If these values do not agree, then either:
- the runner summary logic and notebook logic are inconsistent
- the wrong arrays are being read
- the output keys are mismatched

### If something looks wrong
Large differences usually point to a real workflow issue and should be investigated before trusting sweep results.


In [ ]:
iota_days = INTERVAL_WEEKS * 7

recalc = {
    'C_trough_ugml': float(pk_values[-1]),
    'C_avg_ugml': float(np.trapz(pk_values, t_days) / iota_days),
    'C_max_ugml': float(np.max(pk_values)),
}

if pd_values is not None:
    recalc[f'{pd_key}_trough'] = float(pd_values[-1])

rows = []
for metric, value in recalc.items():
    saved_value = summary_json.get(metric, np.nan)
    rows.append({
        'metric': metric,
        'saved': saved_value,
        'recomputed': value,
        'diff': value - saved_value if pd.notna(saved_value) else np.nan,
    })

compare = pd.DataFrame(rows)
compare


## 8. Optional numerical checks

This section reruns the same scenario with modified numerical settings to check whether the output is reasonably robust.

It compares three things:
- **`dt_days` sensitivity**: finer or coarser output grid
- **solver method sensitivity**: for example `Radau` vs `BDF`
    - All available Models run on `Radau` 
- **tolerance sensitivity**: stricter or looser solver tolerances

### What you should expect
A few small result tables. The metrics should usually stay fairly similar across settings.

### When to worry
Investigate further if small changes in solver settings cause large changes in:
- `C_trough_ugml`
- `C_avg_ugml`
- `C_max_ugml`
- `<pd_key>_trough`

That can indicate stiffness issues, unstable equations, discontinuities, or a model that is too sensitive to the chosen numerical setup.


In [ ]:
# -------------------------
# INPUT — edit only here
# -------------------------

DT_VALUES = [0.25, 0.1, 0.05]
SOLVER_METHODS = ['Radau', 'BDF']
TOLERANCE_PAIRS = [
    (1e-6, 1e-9),
    (1e-7, 1e-10),
]

# -------------------------
# RUN SOLVER SETTING CHECK
# -------------------------

dose = DOSE_MGKG
interval = INTERVAL_WEEKS
cfg_base = load_config(config_path)

# A) dt_days sensitivity
rows_dt = []
for dt in DT_VALUES:
    cfg_test = json.loads(json.dumps(cfg_base))
    cfg_test['simulation']['dt_days'] = dt
    out_dir = repo_root / 'results' / '_notebook_validation' / model_name / f'dt_{dt}'
    s = run_one(cfg_test, dose, interval, out_dir, param_overrides=PARAM_OVERRIDES)
    s['dt_days'] = dt
    rows_dt.append(s)

display(pd.DataFrame(rows_dt)[['dt_days', 'C_trough_ugml', 'C_avg_ugml', 'C_max_ugml'] + ([f'{pd_key}_trough'] if pd_key else [])])

# B) solver sensitivity
rows_solver = []
for method in SOLVER_METHODS:
    cfg_test = json.loads(json.dumps(cfg_base))
    cfg_test['simulation']['solver']['method'] = method
    out_dir = repo_root / 'results' / '_notebook_validation' / model_name / f'solver_{method}'
    s = run_one(cfg_test, dose, interval, out_dir, param_overrides=PARAM_OVERRIDES)
    s['method'] = method
    rows_solver.append(s)

display(
    pd.DataFrame(rows_solver)[
        ['method', 'C_trough_ugml', 'C_avg_ugml', 'C_max_ugml']
        + ([f'{pd_key}_trough'] if pd_key else [])
    ]
)

# C) tolerance sensitivity
rows_tol = []
for rtol, atol in TOLERANCE_PAIRS:
    cfg_test = json.loads(json.dumps(cfg_base))
    cfg_test['simulation']['solver']['rtol'] = float(rtol)
    cfg_test['simulation']['solver']['atol'] = float(atol)
    out_dir = repo_root / 'results' / '_notebook_validation' / model_name / f'rtol_{rtol}_atol_{atol}'
    s = run_one(cfg_test, dose, interval, out_dir, param_overrides=PARAM_OVERRIDES)
    s['rtol'] = rtol
    s['atol'] = atol
    rows_tol.append(s)

display(pd.DataFrame(rows_tol)[['rtol', 'atol', 'C_trough_ugml', 'C_avg_ugml', 'C_max_ugml'] + ([f'{pd_key}_trough'] if pd_key else [])])
